In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Raghav\\Desktop\\imcaption\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Raghav\\Desktop\\imcaption'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    captions_file: Path
    vectorizer_path: Path
    train_images_captions_path: Path
    test_images_captions_path: Path

    SPLIT_SIZE: float
    RANDOM_STATE: int


In [6]:
from imgCaption.constants import *
from imgCaption.utils.common import read_yaml,create_directories,save_pkl_file

In [ ]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:

        config = self.config.data_transformation
        params = self.params

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
                                                root_dir=Path(config.root_dir),
                                                captions_file=Path(config.captions_file),
                                                vectorizer_path=Path(config.vectorizer_path),
                                                train_images_captions_path=Path(config.train_images_captions_path),
                                                test_images_captions_path=Path(config.test_images_captions_path),
                                                SPLIT_SIZE=params.SPLIT_SIZE,
                                                RANDOM_STATE=params.RANDOM_STATE
                                                )

        return data_transformation_config

In [8]:
import os
import re
import json
import pickle
import yaml
import numpy as np
import pandas as pd
from imgCaption import logger
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization

In [9]:
class DataTransformation:
    def __init__(self,config : DataTransformationConfig):
        self.config = config

    def captions_preprocessing(self):
        df = pd.read_csv(self.config.captions_file, sep=",")
        captions = list(df['caption'])
        captions = [
        re.sub(r'\s+', ' ', 
        re.sub(r'\d+', '', 
        re.sub(r'[^\w\s]', '', cap)
        )).strip().lower() 
        for cap in captions]
        captions = list(map(lambda cap: f"startseq {cap} endseq", captions))
        df['captions_tok'] = captions
        logger.info("Data frame created successfully")
        return df

    def split_images(self,df):
        images = list(set(df['image']))
        train_images,test_images=train_test_split(images,test_size=self.config.SPLIT_SIZE,random_state=self.config.RANDOM_STATE)
        logger.info(f"Train images : {len(train_images)}")
        logger.info(f"Test images : {len(test_images)}")
        return train_images,test_images

    def vectorizer(self,df):
        captions = list(df['captions_tok'])
        vectorizer=TextVectorization()
        vectorizer.adapt(captions)
        vocab_size=vectorizer.vocabulary_size()
        max_length=vectorizer(captions).shape[1]
        logger.info(f"Vocab Size : {vocab_size}")
        logger.info(f"Max length of captions : {max_length}")
        existing_params = read_yaml(PARAMS_FILE_PATH)
        existing_params.update({"VOCAB_SIZE": vocab_size, "MAX_LENGTH": max_length})
        with open(PARAMS_FILE_PATH, "w") as file:
            yaml.safe_dump(existing_params.to_dict(), file, sort_keys=False)
        logger.info("Params updated Sucessfully")
        vectorizer_data = {"config": vectorizer.get_config(),
                            "weights": vectorizer.get_weights()}
        save_pkl_file(vectorizer_data, self.config.vectorizer_path)
        logger.info("Vectorizer Saved Sucessfully")

    @staticmethod
    def build_image_caption_map(df,images,output_path:Path):
        image_caption_map = {}
        filtered_df = df[df["image"].isin(images)]
        image_caption_map=filtered_df.groupby("image")["captions_tok"].apply(list).to_dict()
        save_pkl_file(image_caption_map,output_path)
        logger.info("Image captions dict saved")
        return image_caption_map

In [ ]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    df=data_transformation.captions_preprocessing()
    data_transformation.vectorizer(df)
    train_img,test_img=data_transformation.split_images(df)
    data_transformation.build_image_caption_map(df,train_img,data_transformation_config.train_images_captions_path)
    data_transformation.build_image_caption_map(df,test_img,data_transformation_config.test_images_captions_path)
except Exception as e:
    raise e

[2026-07-07 12:06:34,124: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-07 12:06:34,131: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-07 12:06:34,133: INFO: common: created directory at: artifacts]
[2026-07-07 12:06:34,136: INFO: common: created directory at: artifacts/data_transformation]
[2026-07-07 12:06:34,733: INFO: 314189205: Data frame created successfully]
[2026-07-07 12:06:36,684: INFO: 314189205: Vocab Size : 8782]
[2026-07-07 12:06:36,684: INFO: 314189205: Max length of captions : 37]
[2026-07-07 12:06:36,691: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-07 12:06:36,696: INFO: 314189205: Params updated Sucessfully]
[2026-07-07 12:06:36,700: INFO: common: Pickle file saved at: artifacts\data_transformation\vectorizer_data.pkl]
[2026-07-07 12:06:36,704: INFO: 314189205: Vectorizer Saved Sucessfully]
[2026-07-07 12:06:36,711: INFO: 314189205: Train images : 6472]
[2026-07-07 12:06:36,721: INFO: 314189205: T